In [1]:
import numpy as np

from tqdm import tqdm, trange

import torch
import torch.nn as nn
from torch.optim import Adam
from torch.nn import CrossEntropyLoss
from torch.utils.data import DataLoader

from torchvision.transforms import ToTensor
from torchvision.datasets.mnist import MNIST

np.random.seed(0)
torch.manual_seed(0)

In [5]:
import torch

print("GPU disponible :", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU utilisé :", torch.cuda.get_device_name(0))

GPU disponible : True
GPU utilisé : Tesla T4


In [6]:
class MyViT(nn.Module):
    def __init__(
        self,
        chw=(1, 28, 28),
        n_patches=7,
        n_blocks=2,
        hidden_d=8,
        n_heads=2,
        out_d=10
    ):
        super().__init__()

        c, h, w = chw
        assert h % n_patches == 0 and w % n_patches == 0
        assert hidden_d % n_heads == 0

        self.n_patches = n_patches
        self.patch_size = (h // n_patches, w // n_patches)
        self.hidden_d = hidden_d

        # Projection de chaque patch vers un vecteur
        patch_dim = c * self.patch_size[0] * self.patch_size[1]
        self.input_d = nn.Linear(patch_dim, hidden_d)

        # Token CLS et embeddings positionnels
        self.cls_token = nn.Parameter(torch.randn(1, 1, hidden_d))
        self.pos_embed = nn.Parameter(
            torch.randn(1, n_patches * n_patches + 1, hidden_d)
        )

        # Blocs Transformer
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=hidden_d,
            nhead=n_heads,
            dim_feedforward=4 * hidden_d,
            batch_first=True
        )
        self.transformer = nn.TransformerEncoder(
            encoder_layer,
            num_layers=n_blocks
        )

        # Classification en 10 chiffres
        self.mlp_head = nn.Linear(hidden_d, out_d)

    def forward(self, x):
        batch_size = x.shape[0]
        ph, pw = self.patch_size

        # [B, C, H, W] -> [B, nombre de patchs, dimension d'un patch]
        x = x.unfold(2, ph, ph).unfold(3, pw, pw)
        x = x.permute(0, 2, 3, 1, 4, 5).contiguous()
        x = x.reshape(batch_size, self.n_patches ** 2, -1)

        # Embedding des patchs
        x = self.input_d(x)

        # Ajout du token CLS et des positions
        cls = self.cls_token.expand(batch_size, -1, -1)
        x = torch.cat((cls, x), dim=1)
        x = x + self.pos_embed

        # Encodeur Transformer
        x = self.transformer(x)

        # Le token CLS représente l'image
        x = x[:, 0]
        return self.mlp_head(x)

In [7]:

import time
from sklearn.metrics import f1_score, classification_report

def main():
    transform = ToTensor()

    train_set = MNIST(
        root="./datasets",
        train=True,
        download=True,
        transform=transform
    )
    test_set = MNIST(
        root="./datasets",
        train=False,
        download=True,
        transform=transform
    )

    train_loader = DataLoader(
        train_set, shuffle=True, batch_size=128
    )
    test_loader = DataLoader(
        test_set, shuffle=False, batch_size=128
    )

    device = torch.device(
        "cuda" if torch.cuda.is_available() else "cpu"
    )
    print("Using device:", device)

    if torch.cuda.is_available():
        print("GPU:", torch.cuda.get_device_name(0))

    model = MyViT(
        (1, 28, 28),
        n_patches=7,
        n_blocks=2,
        hidden_d=8,
        n_heads=2,
        out_d=10
    ).to(device)

    N_EPOCHS = 5
    LR = 0.005

    optimizer = Adam(model.parameters(), lr=LR)
    criterion = CrossEntropyLoss()

    start_time = time.time()

    # Entraînement
    for epoch in trange(N_EPOCHS, desc="Training"):
        model.train()
        train_loss = 0.0

        for x, y in tqdm(
            train_loader,
            desc=f"Epoch {epoch + 1} in training",
            leave=False
        ):
            x, y = x.to(device), y.to(device)

            optimizer.zero_grad()
            y_hat = model(x)
            loss = criterion(y_hat, y)
            loss.backward()
            optimizer.step()

            train_loss += loss.item() / len(train_loader)

        print(
            f"Epoch {epoch + 1}/{N_EPOCHS} "
            f"loss: {train_loss:.4f}"
        )

    training_time = time.time() - start_time

    # Évaluation
    model.eval()
    true_labels = []
    predictions = []
    test_loss = 0.0

    with torch.no_grad():
        for x, y in tqdm(test_loader, desc="Testing"):
            x, y = x.to(device), y.to(device)

            y_hat = model(x)
            loss = criterion(y_hat, y)
            test_loss += loss.item() / len(test_loader)

            predicted = y_hat.argmax(dim=1)

            true_labels.extend(y.cpu().tolist())
            predictions.extend(predicted.cpu().tolist())

    accuracy = sum(
        p == t for p, t in zip(predictions, true_labels)
    ) / len(true_labels)

    f1 = f1_score(
        true_labels, predictions,
        average="weighted", zero_division=0
    )

    print(f"Test loss: {test_loss:.4f}")
    print(f"Test accuracy: {accuracy * 100:.2f}%")
    print(f"F1-score: {f1:.4f}")
    print(f"Training time: {training_time:.1f} seconds")

    print(classification_report(
        true_labels, predictions, digits=4, zero_division=0
    ))

    torch.save(model.state_dict(), "vit_mnist.pth")
    print("Modèle sauvegardé : vit_mnist.pth")

if __name__ == "__main__":
    main()


Using device: cuda
GPU: Tesla T4


Training:  20%|██        | 1/5 [00:11<00:46, 11.64s/it]

Epoch 1/5 loss: 1.5503



Training:  40%|████      | 2/5 [00:22<00:33, 11.24s/it]

Epoch 2/5 loss: 0.9121


...

Epoch 4/5 loss: 0.6502



Training: 100%|██████████| 5/5 [00:53<00:00, 10.76s/it]


Epoch 5/5 loss: 0.5958


Testing: 100%|██████████| 79/79 [00:01<00:00, 64.70it/s]

Test loss: 0.4141
Test accuracy: 86.41%
F1-score: 0.8627
Training time: 53.8 seconds
              precision    recall  f1-score   support

           0     0.9207    0.9592    0.9395       980
           1     0.9244    0.9797    0.9512      1135
           2     0.9055    0.8547    0.8794      1032
           3     0.7547    0.9198    0.8291      1010
           4     0.9048    0.7454    0.8174       982
           5     0.8692    0.6334    0.7328       892
           6     0.9525    0.8998    0.9254       958
           7     0.8858    0.8979    0.8918      1028
           8     0.8082    0.8655    0.8359       974
           9     0.7602    0.8454    0.8006      1009

    accuracy                         0.8641     10000
   macro avg     0.8686    0.8601    0.8603     10000
weighted avg     0.8689    0.8641    0.8627     10000

Modèle sauvegardé : vit_mnist.pth


Accuracy = 86,41 % : le modèle a correctement classé 8 641 images sur 10 000. Il s'est donc trompé sur 1 359 images.



F1-score pondéré = 0,8627 : ce score combine la précision et le rappel en tenant compte du nombre d'images de chaque classe. Il est cohérent avec l'accuracy obtenue.



Loss de test = 0,4141 : elle mesure l'écart entre les scores prédits et les véritables classes. Plus cette valeur est faible, mieux c'est, lorsque l'on compare des expériences utilisant la même fonction de perte.



Temps = 53,8 secondes : l'entraînement a été rapide, mais il n'a duré que cinq époques avec une architecture ViT très légère (hidden_d=8, n_blocks=2).

Les chiffres 1, 0 et 6 sont les mieux reconnus, avec des F1-scores respectifs de 0,9512, 0,9395 et 0,9254.

Le chiffre 5 est le plus difficile à reconnaître : son F1-score est de 0,7328 et son rappel est de 0,6334. Cela signifie que le modèle ne retrouve qu'environ 63,34 % des véritables images de la classe 5.

Le chiffre 4 présente également un rappel assez faible (0,7454). Ces résultats peuvent s'expliquer par la diversité des écritures manuscrites et par la capacité limitée de l'architecture utilisée.